# Model cookbook

A one-page, indexed gallery of pulsar-timing model recipes — from a bare white-noise model up to a full PTA model with a Hellings–Downs background, a decentered parameterisation, and a continuous-wave signal.

Each recipe is a small function that assembles a Discovery likelihood from the public API. **These are the exact models the parity test-suite exercises** (`tests/metamatrix/`): they are defined once in `discovery.recipes` and imported both there and by the tests, so every recipe on this page is covered by a test that asserts the `matrix` and `metamath` kernel backends agree.

Every recipe works unchanged under either backend — flip between them with `ds.config(kernels='matrix')` and `ds.config(kernels='metamath')` (see the last section).


> **These recipes are illustrative, not a configurable API.** Each takes only a pulsar (or list of pulsars) — there are no keyword knobs — so treat them as starting points to copy and adapt, not components to parametrise. The small shared helpers they reuse (`_psl_skeleton`, `_psl_with_rn`, `_toy_delay`) are shown in **Building blocks** below, so every ingredient is in one place.


In [1]:
import inspect
from pathlib import Path

import numpy as np
import jax
jax.config.update('jax_enable_x64', True)

import discovery as ds
import discovery.recipes as R   # the model zoo (also used by the parity tests)
from IPython.display import Markdown, HTML, display

# bundled example pulsars
datapath = Path(ds.__path__[0]) / '../../data'
files = sorted(datapath.glob('v1p1*.feather'))
psr = ds.Pulsar.read_feather(files[0])          # one pulsar, for single-pulsar recipes
psrs = [ds.Pulsar.read_feather(f) for f in files[:3]]   # a few, for multi-pulsar recipes
print(f'single: {psr.name};  array: {[p.name for p in psrs]}')

single: B1855+09;  array: ['B1855+09', 'B1937+21', 'B1953+29']


In [2]:
def show(recipe, arg):
    """Render one recipe: caption (its docstring), source, sampled params, and a logL value."""
    caption = ' '.join((recipe.__doc__ or '').split())
    display(Markdown(f'<a id="recipe-{recipe.__name__}"></a>\n\n#### `{recipe.__name__}`\n\n{caption}'))
    display(Markdown(f"```python\n{inspect.getsource(recipe)}\n```"))

    model = recipe(arg)
    params = list(model.logL.params)
    display(Markdown(f"- **sampled (hyper)parameters** ({len(params)}): "
                     + (', '.join(f'`{p}`' for p in params) if params else '_none_')))
    try:
        np.random.seed(0)
        p0 = ds.sample_uniform(params) if params else {}
        display(Markdown(f"- log-likelihood at a random draw: `{float(model.logL(p0)):.3f}`"))
    except Exception as e:
        display(Markdown(f"- _logL eval skipped ({type(e).__name__}: needs manual params)_"))

def show_src(fn, note=''):
    """Render a shared helper: its source plus a one-line note."""
    display(Markdown(f"**`{fn.__name__}`** — {note}\n\n```python\n{inspect.getsource(fn)}\n```"))


def make_toc():
    """Build the 3-section recipe index as an HTML table, linked to each recipe below."""
    def esc(t): return t.replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')
    sections = [('Single-pulsar — PulsarLikelihood', R.SINGLE_PULSAR),
                ('Multi-pulsar — GlobalLikelihood', R.GLOBAL),
                ('Multi-pulsar — ArrayLikelihood', R.ARRAY)]
    html = []
    for title, recipes in sections:
        html.append(f'<h3>{title}</h3>')
        html.append('<table style="text-align:left"><thead><tr>'
                    '<th style="text-align:left">Recipe</th>'
                    '<th style="text-align:left">What it builds</th></tr></thead><tbody>')
        for r in recipes:
            cap = esc(' '.join((r.__doc__ or '').split()))
            html.append(f'<tr><td style="text-align:left">'
                        f'<a href="#recipe-{r.__name__}"><code>{r.__name__}</code></a></td>'
                        f'<td style="text-align:left">{cap}</td></tr>')
        html.append('</tbody></table>')
    display(HTML('\n'.join(html)))

## Recipe index

Click a recipe to jump to it. Every recipe is in `discovery.recipes` and is parity-tested under both kernel backends.

In [3]:
make_toc()

Recipe,What it builds
measurement_simple,"White noise only, single-backend (efac + t2equad), no selection."
measurement_white,White noise with per-backend efac/equad selection.
ecorr_gp,White noise plus ECORR modelled as a separate Gaussian-process component.
ecorr_sm,ECORR folded into the noise matrix via Sherman-Morrison (+ timing GP wrapper).
meas_timing,White noise plus an (SVD-stabilised) marginalised timing-model GP.
full_rn,Realistic single-pulsar model: white + ECORR-GP + timing + power-law red noise.
full_rn_concat_false,Same as full_rn but with concat=False → chained (nested) Woodbury kernels.
multi_vgp,Two variable GPs: achromatic red noise + a chromatic DM GP (on the DM Fourier basis).
variable_timing,"Timing model as a *variable* GP (coefficients sampled, not marginalised) + RN."
fftcov_2d,Red noise via an FFT-derived dense (2D) covariance basis (makegp_fftcov).


## Building blocks (shared helpers)

The multi-pulsar recipes reuse two small per-pulsar model helpers, and the single-pulsar `delay` recipe uses a toy delay function. Here they are in full — the only pieces the recipes rely on that aren't already inline:

In [4]:
for fn, note in [
    (R._psl_skeleton, 'per-pulsar model for the ArrayLikelihood recipes (no red noise — it lives in the commongp)'),
    (R._psl_with_rn,  'per-pulsar model for the GlobalLikelihood recipes (carries its own intrinsic red noise)'),
    (R._toy_delay,    'deterministic, parameter-free delay used by the single-pulsar `delay` recipe'),
]:
    show_src(fn, note)

**`_psl_skeleton`** — per-pulsar model for the ArrayLikelihood recipes (no red noise — it lives in the commongp)

```python
def _psl_skeleton(psr):
    # per-pulsar model WITHOUT red noise (red noise lives in the commongp)
    return ds.PulsarLikelihood([
        psr.residuals,
        ds.makenoise_measurement(psr, psr.noisedict),
        ds.makegp_ecorr(psr, psr.noisedict),
        ds.makegp_timing(psr, svd=True),
    ])

```

**`_psl_with_rn`** — per-pulsar model for the GlobalLikelihood recipes (carries its own intrinsic red noise)

```python
def _psl_with_rn(psr, T):
    # per-pulsar model carrying its own red noise (for GlobalLikelihood rows)
    return ds.PulsarLikelihood([
        psr.residuals,
        ds.makenoise_measurement(psr, psr.noisedict),
        ds.makegp_ecorr(psr, psr.noisedict),
        ds.makegp_timing(psr, svd=True),
        ds.makegp_fourier(psr, ds.powerlaw, components=30, T=T, name="rednoise"),
    ])

```

**`_toy_delay`** — deterministic, parameter-free delay used by the single-pulsar `delay` recipe

```python
def _toy_delay(toas):
    # deterministic, parameter-free delay (args come only from psr attributes).
    return 1e-9 * jnp.sin(2.0 * jnp.pi * (toas - toas.min()) / 3.16e8)

```

## Single-pulsar models — `PulsarLikelihood`

A {class}`~discovery.likelihood.PulsarLikelihood` is a list whose first entry is the residuals and the rest are noise / GP / delay components. Ordered roughly from simplest to most featureful.

**Functions used** (click through to the API page, which has a `[source]` link to the code): {func}`~discovery.measurement_noise.makenoise_measurement`, {func}`~discovery.measurement_noise.makenoise_measurement_simple`, {func}`~discovery.signals.makegp_ecorr`, {func}`~discovery.signals.makegp_timing`, {func}`~discovery.signals.makegp_fourier`, {func}`~discovery.signals.makegp_fftcov`, {func}`~discovery.signals.makegp_fourier_variance`, {func}`~discovery.signals.makedelay`, {func}`~discovery.signals.dmfourierbasis`; wrapped in {class}`~discovery.likelihood.PulsarLikelihood`.

In [5]:
for recipe in R.SINGLE_PULSAR:
    show(recipe, psr)

<a id="recipe-measurement_simple"></a>

#### `measurement_simple`

White noise only, single-backend (efac + t2equad), no selection.

```python
def measurement_simple(psr):
    """White noise only, single-backend (efac + t2equad), no selection."""
    return ds.PulsarLikelihood([
        psr.residuals,
        ds.makenoise_measurement_simple(psr),
    ])

```

- **sampled (hyper)parameters** (2): `B1855+09_efac`, `B1855+09_log10_t2equad`

- log-likelihood at a random draw: `88934.233`

<a id="recipe-measurement_white"></a>

#### `measurement_white`

White noise with per-backend efac/equad selection.

...

<a id="recipe-fourier_variance_fixed"></a>

#### `fourier_variance_fixed`

A Fourier GP whose prior covariance is supplied directly as a fixed matrix.

```python
def fourier_variance_fixed(psr):
    """A Fourier GP whose prior covariance is supplied directly as a fixed matrix."""
    comps = 10
    argname = f"{psr.name}_fourierGP_variance({comps * 2},{comps * 2})"
    cov = np.diag(np.full(comps * 2, 1e-4))
    return ds.PulsarLikelihood([
        psr.residuals,
        ds.makenoise_measurement(psr, psr.noisedict),
        ds.makegp_timing(psr, svd=True),
        ds.makegp_fourier_variance(psr, components=comps, noisedict={argname: cov}),
    ])

```

/Users/pmeyers/miniforge3/envs/discovery/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: divide by zero encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/pmeyers/miniforge3/envs/discovery/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: overflow encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)
/Users/pmeyers/miniforge3/envs/discovery/lib/python3.13/site-packages/numpy/linalg/_linalg.py:2374: RuntimeWarning: invalid value encountered in slogdet
  sign, logdet = _umath_linalg.slogdet(a, signature=signature)


- **sampled (hyper)parameters** (0): _none_

- log-likelihood at a random draw: `90606.014`

## Multi-pulsar — `GlobalLikelihood`

{class}`~discovery.likelihood.GlobalLikelihood` wraps a list of per-pulsar {class}`~discovery.likelihood.PulsarLikelihood`s and an optional `globalgp` for a signal correlated *across* pulsars (e.g. a Hellings–Downs background).

**Functions used:** {class}`~discovery.likelihood.GlobalLikelihood`, {func}`~discovery.signals.makeglobalgp_fourier`, {func}`~discovery.signals.hd_orf`, {func}`~discovery.signals.monopole_orf`, {func}`~discovery.signals.CompoundGlobalGP`.

In [6]:
for recipe in R.GLOBAL:
    show(recipe, psrs)

<a id="recipe-no_global"></a>

#### `no_global`

Independent pulsars — GlobalLikelihood with no correlated GP (sum of per-psr logL).

```python
def no_global(psrs):
    """Independent pulsars — GlobalLikelihood with no correlated GP (sum of per-psr logL)."""
    T = ds.getspan(psrs)
    return ds.GlobalLikelihood([_psl_with_rn(p, T) for p in psrs])

```

- **sampled (hyper)parameters** (6): `B1855+09_rednoise_gamma`, `B1855+09_rednoise_log10_A`, `B1937+21_rednoise_gamma`, `B1937+21_rednoise_log10_A`, `B1953+29_rednoise_gamma`, `B1953+29_rednoise_log10_A`

- log-likelihood at a random draw: `472034.430`

<a id="recipe-global_hd"></a>

#### `global_hd`

A Hellings-Downs-correlated common GW signal across pulsars (dense 2D prior).

...

- log-likelihood at a random draw: `473743.112`

<a id="recipe-global_compound"></a>

#### `global_compound`

Two correlated global GPs at once (HD + monopole) via a globalgp list (CompoundGlobalGP).

```python
def global_compound(psrs):
    """Two correlated global GPs at once (HD + monopole) via a globalgp list (CompoundGlobalGP)."""
    T = ds.getspan(psrs)
    return ds.GlobalLikelihood(
        [_psl_with_rn(p, T) for p in psrs],
        globalgp=[
            ds.makeglobalgp_fourier(psrs, ds.powerlaw, ds.hd_orf,
                                    components=14, T=T, name="gw"),
            ds.makeglobalgp_fourier(psrs, ds.powerlaw, ds.monopole_orf,
                                    components=14, T=T, name="gw_mono"),
        ],
    )

```

- **sampled (hyper)parameters** (10): `B1855+09_rednoise_gamma`, `B1855+09_rednoise_log10_A`, `B1937+21_rednoise_gamma`, `B1937+21_rednoise_log10_A`, `B1953+29_rednoise_gamma`, `B1953+29_rednoise_log10_A`, `gw_gamma`, `gw_log10_A`, `gw_mono_gamma`, `gw_mono_log10_A`

- log-likelihood at a random draw: `473689.763`

## Multi-pulsar — `ArrayLikelihood`

{class}`~discovery.likelihood.ArrayLikelihood` is the vectorised multi-pulsar likelihood. It adds `commongp` (a shared-basis process across pulsars), `globalgp` (cross-pulsar correlated), `decenter` (whitened-coefficient sampling), per-GP `means`, and `extsignals` (deterministic signals on their own basis, e.g. a continuous wave).

**Functions used:** {class}`~discovery.likelihood.ArrayLikelihood`, {func}`~discovery.signals.makecommongp_fourier`, {func}`~discovery.signals.make_combined_crn`, {func}`~discovery.deterministic.makecw_extsignal`.

In [7]:
for recipe in R.ARRAY:
    show(recipe, psrs)

<a id="recipe-no_common"></a>

#### `no_common`

ArrayLikelihood with per-pulsar red noise inline, no shared/correlated GP.

```python
def no_common(psrs):
    """ArrayLikelihood with per-pulsar red noise inline, no shared/correlated GP."""
    T = ds.getspan(psrs)
    return ds.ArrayLikelihood([
        ds.PulsarLikelihood([
            psr.residuals,
            ds.makenoise_measurement(psr, psr.noisedict),
            ds.makegp_ecorr(psr, psr.noisedict),
            ds.makegp_timing(psr, svd=True),
            ds.makegp_fourier(psr, ds.powerlaw, components=30, T=T, name="rednoise"),
        ]) for psr in psrs
    ])

```

- **sampled (hyper)parameters** (6): `B1855+09_rednoise_gamma`, `B1855+09_rednoise_log10_A`, `B1937+21_rednoise_gamma`, `B1937+21_rednoise_log10_A`, `B1953+29_rednoise_gamma`, `B1953+29_rednoise_log10_A`

- log-likelihood at a random draw: `472034.430`

<a id="recipe-intrinsic_rn"></a>

#### `intrinsic_rn`

Per-pulsar intrinsic red noise on a shared Fourier basis (vectorised; independent amplitudes per pulsar).

...

- _logL eval skipped (KeyError: needs manual params)_

<a id="recipe-extsignal_cw"></a>

#### `extsignal_cw`

Intrinsic red noise plus a continuous-wave deterministic signal on its own basis.

```python
def extsignal_cw(psrs):
    """Intrinsic red noise plus a continuous-wave deterministic signal on its own basis."""
    T = ds.getspan(psrs)
    return ds.ArrayLikelihood(
        [_psl_skeleton(p) for p in psrs],
        commongp=ds.makecommongp_fourier(psrs, ds.powerlaw, components=30,
                                         T=T, name="rednoise"),
        extsignals=[
            ds.makecw_extsignal(psrs, components=50, T=T, pulsarterm=True, name="cw"),
        ],
    )

```

- **sampled (hyper)parameters** (6): `B1855+09_rednoise_gamma`, `B1855+09_rednoise_log10_A`, `B1937+21_rednoise_gamma`, `B1937+21_rednoise_log10_A`, `B1953+29_rednoise_gamma`, `B1953+29_rednoise_log10_A`

- log-likelihood at a random draw: `472034.430`

## Switching kernel backends

The same recipe runs on either kernel implementation; `ds.config(kernels=...)` selects it before you build the model. The two agree to numerical precision (this equivalence is what the parity suite certifies).

In [8]:
np.random.seed(0)
p0 = ds.sample_uniform(R.full_rn(psr).logL.params)

ds.config(kernels='matrix')
v_matrix = float(R.full_rn(psr).logL(p0))

ds.config(kernels='metamath')
v_metamath = float(R.full_rn(psr).logL(p0))

ds.config(kernels='matrix')   # restore default
print(f'full_rn logL  matrix={v_matrix:.6f}  metamath={v_metamath:.6f}  diff={abs(v_matrix-v_metamath):.2e}')

full_rn logL  matrix=91087.407075  metamath=91087.407075  diff=4.37e-11
